# Predicting Future Urgent Orders of a Logistics Company using Recurrent Neural Networks

# By: Samuel Reano

In this Deep Learning project, we used a time-series dataset from a Brazilian logistics company. We created two models to predict future urgent orders, Naive baseline, and a stacked Gated Recurrent Unit (GRU). In the Naive baseline model, we use the actual observations corresponding to two weeks prior as the prediction for a variable and to find predictions for test samples.  We calculated the average root mean squared error and the mean absolute error over the test samples. In the stacked Gated Recurrent Unit model, we trained it to predict the next 7 days urgent orders using a look back of 14 days. Again, we calculated the average root mea squared error and the mean absolute error over the test samples. We compared the results of both models and concluded that the trained stacked Gated Recurrent Unit model performed better than the Naive baseline model. All of this was done using Python on Google Colaboratory.

The dataset used in this project came from the following website: https://archive.ics.uci.edu/dataset/409/daily+demand+forecasting+orders

## Preparation

In [ ]:
import pandas as pd
import numpy as np
from numpy import hstack
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.preprocessing.text import Tokenizer
from keras.preprocessing import sequence
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, Embedding
from keras import optimizers
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage.interpolation import shift
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from keras.callbacks import EarlyStopping
from keras.optimizers import RMSprop, Adam
from keras.activations import swish

<ipython-input-30-7bf8915dea08>:14: DeprecationWarning: Please use `shift` from the `scipy.ndimage` namespace, the `scipy.ndimage.interpolation` namespace is deprecated.
  from scipy.ndimage.interpolation import shift


## Naive Baseline Model

In [ ]:
# The csv file had been downloaded from the website. Open the csv file
brazil = pd.read_csv("Daily_Demand_Forecasting_Orders.csv", sep=";")
print(brazil)

    Week of the month (first week, second, third, fourth or fifth week  \
0                                                   1                    
1                                                   1                    
2                                                   1                    
3                                                   2                    
4                                                   2                    
5                                                   2                    
6                                                   2                    
7                                                   2                    
8                                                   3                    
9                                                   3                    
10                                                  3                    
11                                                  3                    
12                                    

In [ ]:
# Extract the 'Urgent order' column
urgent_orders = brazil['Urgent order'].values

# Define the prediction horizon
prediction_horizon = 7

# For the naive baseline, use the actual observations corresponding to two weeks prior
naive_baseline = urgent_orders[:-14]

# Define the test samples
test_samples = urgent_orders[-prediction_horizon:]

# Calculate the naive baseline predictions for the test samples
naive_predictions = naive_baseline[-prediction_horizon:]

# Calculate the average RMSE and MAE over the test sample
rmse1 = np.sqrt(mean_squared_error(test_samples, naive_predictions))
mae1 = mean_absolute_error(test_samples, naive_predictions)

print(f'Average RMSE: {rmse1}')
print(f'Average MAE: {mae1}')


Average RMSE: 40.85560438387161
Average MAE: 33.94414285714286


## Gated Recurrent Unit Model

In [ ]:
# Define a function to create dataset with look back
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back)]
        dataX.append(a)
        dataY.append(dataset[i + look_back])
    return np.array(dataX), np.array(dataY)

# Normalize the dataset
urgent_orders = urgent_orders.astype('float32')

# Reshape into X=t and Y=t+1
look_back = 14
dataX, dataY = create_dataset(urgent_orders, look_back)

# Split into train and test sets
train_size = len(dataX) - 7  # Use last 7 samples for testing
trainX, testX = dataX[:train_size], dataX[train_size:]
trainY, testY = dataY[:train_size], dataY[train_size:]

# Reshape input to be [samples, time steps, features]
trainX = np.reshape(trainX, (trainX.shape[0], 1, trainX.shape[1]))
testX = np.reshape(testX, (testX.shape[0], 1, testX.shape[1]))

# Create and fit the GRU network
model4 = Sequential()
model4.add(GRU(50, activation=swish, return_sequences=True, input_shape=(1, look_back)))
model4.add(GRU(16, activation=swish))
model4.add(Dense(1))
model4.compile(loss='mean_squared_error', optimizer=Adam())
model4.fit(trainX, trainY, epochs=100, batch_size=128, validation_split=0.2)

# Make predictions
trainPredict = model4.predict(trainX)
testPredict = model4.predict(testX)

# Calculate root mean squared error
trainScore1 = np.sqrt(mean_squared_error(trainY, trainPredict[:,0]))
print('Train Score: %.2f RMSE' % (trainScore1))
testScore1 = np.sqrt(mean_squared_error(testY, testPredict[:,0]))
print('Test Score: %.2f RMSE' % (testScore1))

# Calculate mean absolute error
trainScore2 = mean_absolute_error(trainY, trainPredict[:,0])
print('Train Score: %.2f MAE' % (trainScore2))
testScore2 = mean_absolute_error(testY, testPredict[:,0])
print('Test Score: %.2f MAE' % (testScore2))


Epoch 1/100
1/1 [==============================] - 6s 6s/step - loss: 17865.3672 - val_loss: 15718.1514
Epoch 2/100
1/1 [==============================] - 0s 89ms/step - loss: 15913.3213 - val_loss: 14331.6162
Epoch 3/100
1/1 [==============================] - 0s 70ms/step - loss: 14635.8086 - val_loss: 13442.9434
Epoch 4/100
1/1 [==============================] - 0s 100ms/step - loss: 13830.5791 - val_loss: 12815.9824
Epoch 5/100
1/1 [==============================] - 0s 80ms/step - loss: 12902.8096 - val_loss: 12185.4297
Epoch 6/100
1/1 [==============================] - 0s 80ms/step - loss: 11900.0127 - val_loss: 11330.2305
Epoch 7/100
1/1 [==============================] - 0s 83ms/step - loss: 10893.8682 - val_loss: 10315.1973
Epoch 8/100
1/1 [==============================] - 0s 83ms/step - loss: 9993.6084 - val_loss: 9240.9482
Epoch 9/100
1/1 [==============================] - 0s 82ms/step - loss: 9231.7119 - val_loss: 8293.4541
Epoch 10/100
1/1 [==============================] -

##### **Results of GRU:**
##### Train Score is 23.61 RMSE
##### Test Score is 30.58 RMSE
##### Train Score is 19.95 MAE
##### Test Score is 25.63 MAE

##### **Results of Naive baseline:**
##### Average RMSE is 40.85560438387161
##### Average MAE is 33.94414285714286


##### The GRU model performed better than the Naive baseline model. The RMSE for the GRU model on the test set is 30.58, while the average RMSE for the Naive baseline is 40.86. Since RMSE is a measure of error and lower values are better, the GRU model has a lower error rate. The MAE for the GRU model on the test set is 25.63, while the average MAE for the Naive baseline is 33.94. Since MAE is a measure of error and lower values are better, the GRU model has a lower error rate.